In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
os.makedirs('noura_foods_project/data/silver',exist_ok=True)

In [3]:
BASE_DIR=r"C:\Users\gh81a\Desktop\PythonProjectnoura\noura_foods_project\data\raw"

In [4]:
file_names = {
    "Customers": "raw_customers.csv",
    "Orders": "raw_orders.csv","Order_Items": "raw_order_items.csv",
    "Products": "raw_products.csv","Campaigns": "raw_campaigns.csv",
    "Stores": "raw_stores.csv","Economy": "raw_economic_factors.csv"}

In [5]:
dataframes={}
for name,file in file_names.items():
    path=os.path.join(BASE_DIR,file)
    dataframes[name]=pd.read_csv(path)

In [6]:
df_orders = dataframes["Orders"].copy()
df_items = dataframes["Order_Items"].copy()
df_customers = dataframes["Customers"].copy()

In [7]:
def check_data_quality(df_dict):
    report=[]
    for name,df in df_dict.items():
        rows = df.shape[0]
        cols = df.shape[1]
        duplicates = df.duplicated().sum()
        null_cols = df.columns[df.isnull().any()].tolist()
        total_nulls = df.isnull().sum().sum()

        report.append({
            "Table Name": name,"Rows": rows,"Columns": cols,
            "Duplicates": duplicates,"Missing Values (Nulls)": total_nulls,
            "Columns with Nulls": ", ".join(null_cols) if null_cols else "None"})
    return pd.DataFrame(report)
health_report=check_data_quality(dataframes)
display(health_report)



,Table Name,Rows,Columns,Duplicates,Missing Values (Nulls),Columns with Nulls
0,Customers,5000,11,0,0,None
1,Orders,92767,6,0,72504,"TotalAmount, CampaignID"
2,Order_Items,503678,6,0,0,None
3,Products,1584,8,0,0,None
4,Campaigns,150,4,0,0,None
5,Stores,100,5,0,0,None
6,Economy,95,6,0,0,None


In [8]:
def outliers_rep(df_dict):
    report=[]
    for name,df in df_dict.items():
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
        for col in numeric_cols:
            if 'ID' in col or col in ['Year', 'Month', 'Day', 'Age']:
                continue
            Q1=df[col].quantile(0.25)
            Q3=df[col].quantile(0.75)
            IQR=Q3-Q1
            lower_bound=Q1-(1.5 * IQR)
            upper_bound=Q3+(1.5 * IQR)
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            outlier_count = len(outliers)
            if outlier_count > 0:
                outlier_percent = round((outlier_count / len(df)) * 100, 2)
                report.append({"Table Name": name,
                    "Column": col,"Total Outliers": outlier_count,
                    "Percentage (%)":outlier_percent,"Action Logic":"Business Review"})

    return pd.DataFrame(report)
outlier_report =outliers_rep(dataframes)
display(outlier_report)


,Table Name,Column,Total Outliers,Percentage (%),Action Logic
0,Orders,TotalAmount,421,0.45,Business Review
1,Order_Items,Quantity,2518,0.50,Business Review
2,Order_Items,LineTotal,3605,0.72,Business Review
3,Economy,ConsumerConfidenceIndex,5,5.26,Business Review


In [9]:
display(df_customers['Occupation'].value_counts())

Occupation
Teacher                 535
Accountant              524
Business Owner          517
Software Developer      517
Sales Representative    509
Student                 504
Engineer                499
Employee                481
Freelancer              464
Nurse                   450
Name: count, dtype: int64

همانطور که خروجی بالا نشان می‌دهد، تمام مشتریان افراد حقیقی هستند (دانشجو، پرستار و غیره).

In [10]:
p99_per_product = df_items.groupby('ProductID')['Quantity'].transform(lambda x: x.quantile(0.99))
df_items['Quantity'] = np.where(df_items['Quantity'] > p99_per_product, p99_per_product, df_items['Quantity'])

In [11]:
df_items['LineTotal'] = df_items['Quantity'] * df_items['UnitPrice']

In [12]:
df_orders['CampaignID']=df_orders['CampaignID'].fillna('Organic')

In [13]:
real_totals = df_items.groupby('OrderID')['LineTotal'].sum().reset_index()
real_totals.rename(columns={'LineTotal':'CalculatedTotal'}, inplace=True)
df_orders = df_orders.merge(real_totals,on='OrderID',how='left')
df_orders['TotalAmount']=df_orders['CalculatedTotal']
df_orders.drop(columns=['CalculatedTotal'], inplace=True)

In [14]:
dataframes['Orders']= df_orders
dataframes['Order_Items']= df_items
dataframes['Customers']= df_customers
SILVER_DIR = r"C:\Users\gh81a\Desktop\PythonProjectnoura\noura_foods_project\data\silver"
for name, df in dataframes.items():
    file_name = f"{name.lower()}_clean.csv"
    df.to_csv(os.path.join(SILVER_DIR, file_name),index=False)

Phase 2: Building the Gold Layer (Star Schema)

In [15]:
import shutil

In [17]:
SILVER_DIR =r"C:\Users\gh81a\Desktop\PythonProjectnoura\noura_foods_project\data\silver"
GOLD_DIR =r"C:\Users\gh81a\Desktop\PythonProjectnoura\noura_foods_project\data\gold"
os.makedirs(GOLD_DIR,exist_ok=True)
df_orders =pd.read_csv(os.path.join(SILVER_DIR, "orders_clean.csv"))
df_items =pd.read_csv(os.path.join(SILVER_DIR, "order_items_clean.csv"))
# (Fact_Sales)
cols_to_add =['OrderID', 'CustomerID', 'StoreID', 'OrderDate', 'CampaignID']
fact_sales =pd.merge(df_items, df_orders[cols_to_add], on='OrderID', how='inner')
fact_sales.to_csv(os.path.join(GOLD_DIR, "Fact_Sales.csv"), index=False)

#(Dim)
dims={"customers_clean.csv": "Dim_Customers.csv","products_clean.csv": "Dim_Products.csv",
    "stores_clean.csv": "Dim_Stores.csv","campaigns_clean.csv": "Dim_Campaigns.csv","economy_clean.csv": "Dim_Economy.csv"}
for silver_file,gold_file in dims.items():
    shutil.copy(os.path.join(SILVER_DIR,silver_file),os.path.join(GOLD_DIR,gold_file))
